# OpenPlaque — Independent Coronary Model Anatomy Validation v1

This notebook runs an **independently trained ImageCAS-X coronary-lumen model** on the cached UCLA Series 7 CCTA and compares that prediction with the frozen OpenPlaque LAD, RCA, C6 and C7 paths.

The external model is a binary lumen observer. It is not allowed to modify the frozen master or promote clinical LM/LCX/OM identity.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Reuse/cache controls — intentionally immediately after Drive mount
REUSE_VALID_PREDICTION = True
FORCE_EXTERNAL_INFERENCE = False

DRIVE_ROOT = "/content/drive/MyDrive/OpenPlaque"
OUT = f"{DRIVE_ROOT}/Independent_Coronary_Model_Anatomy_Validation_v1"
print("Output:", OUT)

In [ ]:
import os, shutil, sys, subprocess
from pathlib import Path

OPENPLAQUE_PIN = "efd1a5c0dd39741885d6863d68070f7c9de7a5b5"
OPENPLAQUE_BRANCH = "independent-coronary-model-anatomy-validation-from-main"
IMAGECAS_X_PIN = "dbc7343187adf45ca9306dc3460eebc70f92b204"

for p in ["/content/OpenPlaque", "/content/ImageCAS-X"]:
    if Path(p).exists():
        shutil.rmtree(p)

!git clone -q --branch {OPENPLAQUE_BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {OPENPLAQUE_PIN}
!git clone -q https://github.com/kitbransby/ImageCAS-X.git /content/ImageCAS-X
!git -C /content/ImageCAS-X checkout -q {IMAGECAS_X_PIN}

%pip install -q /content/OpenPlaque
%pip install -q /content/ImageCAS-X
%pip install -q requests

# Purge any stale OpenPlaque imports before science.
for name in list(sys.modules):
    if name == "openplaque" or name.startswith("openplaque."):
        del sys.modules[name]

print("OpenPlaque pin:", subprocess.check_output(["git","-C","/content/OpenPlaque","rev-parse","HEAD"], text=True).strip())
print("ImageCAS-X pin:", subprocess.check_output(["git","-C","/content/ImageCAS-X","rev-parse","HEAD"], text=True).strip())

In [ ]:
# Synthetic test + pytest before science
from openplaque.independent_coronary_model_anatomy_validation_v1 import synthetic_self_test
print(synthetic_self_test())
!cd /content/OpenPlaque && pytest -q tests/test_independent_coronary_model_anatomy_validation_v1.py

In [ ]:
from pathlib import Path
import numpy as np
import SimpleITK as sitk
from openplaque.independent_coronary_model_anatomy_validation_v1 import write_source_nifti

# First prove this runtime can write/read a SimpleITK image locally.
smoke = Path("/content/openplaque_sitk_smoke.mha")
tiny = sitk.GetImageFromArray(np.arange(27, dtype=np.int16).reshape(3,3,3))
w = sitk.ImageFileWriter(); w.SetFileName(str(smoke)); w.SetUseCompression(False); w.Execute(tiny)
if not smoke.is_file() or smoke.stat().st_size == 0:
    raise RuntimeError("SimpleITK local MHA smoke test failed")
check = sitk.ReadImage(str(smoke))
assert check.GetSize() == tiny.GetSize()
print("SimpleITK local write/read smoke test passed:", smoke.stat().st_size, "bytes")

out = Path(OUT)
prediction_dir = out / "external_model"
prediction_dir.mkdir(parents=True, exist_ok=True)

# Large transient source image stays in local Colab storage and is uncompressed MHA.
local_input_dir = Path("/content/openplaque_imagecas_input")
local_input_dir.mkdir(parents=True, exist_ok=True)
source_nii = local_input_dir / "ucla.img.mha"
prediction_nii = prediction_dir / "ucla.nii.gz"

write_source_nifti(DRIVE_ROOT, source_nii)
print("Series 7 verified local source image:", source_nii, source_nii.stat().st_size, "bytes")
print("Cached prediction in Drive:", prediction_nii, prediction_nii.exists())

## External ImageCAS-X inference

ImageCAS-X publishes pretrained benchmark weights through Zenodo record **21887809**. The code below discovers the current file list through the Zenodo API and downloads the CAS-Net checkpoint without assuming an unverified filename.

If a valid cached prediction already exists in Drive, it is reused.

In [ ]:
import json, os, re, requests, shutil, zipfile
from pathlib import Path

def acquire_casnet_checkpoint(cache_dir):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    existing = list(cache_dir.rglob("*cas*net*.pt")) + list(cache_dir.rglob("*cas*net*.pth"))
    if existing:
        return existing[0]

    rec = requests.get("https://zenodo.org/api/records/21887809", timeout=60)
    rec.raise_for_status()
    files = rec.json()["files"]
    for f in files:
        print(f["key"], f.get("size"))

    def score(f):
        k=f["key"].lower()
        s=0
        if "cas_net" in k or "cas-net" in k or "casnet" in k: s += 100
        elif "cas" in k: s += 40
        if "weight" in k or "pretrain" in k or "checkpoint" in k: s += 20
        if k.endswith((".pt",".pth")): s += 20
        if k.endswith((".zip",".tar.gz",".tgz")): s += 10
        return s

    candidates=sorted(files,key=score,reverse=True)
    for f in candidates:
        if score(f) <= 0:
            continue
        key=f["key"]
        target=cache_dir/key.replace("/","_")
        if not target.exists():
            url=f.get("links",{}).get("content") or f.get("links",{}).get("self")
            print("Downloading candidate:", key)
            with requests.get(url,stream=True,timeout=180) as r:
                r.raise_for_status()
                with open(target,"wb") as w:
                    for chunk in r.iter_content(8*1024*1024):
                        if chunk: w.write(chunk)
        if target.suffix.lower() in (".pt",".pth") and "cas" in target.name.lower():
            return target
        if zipfile.is_zipfile(target):
            ex=cache_dir/(target.name+"_extracted")
            ex.mkdir(exist_ok=True)
            with zipfile.ZipFile(target) as z:
                z.extractall(ex)
            hits=[p for p in ex.rglob("*") if p.suffix.lower() in (".pt",".pth") and "cas" in p.name.lower()]
            if hits:
                return hits[0]

    raise RuntimeError("Could not identify a CAS-Net checkpoint in Zenodo record 21887809; inspect the printed record file list.")

weight_cache = Path(DRIVE_ROOT)/"Cache"/"ImageCAS_X_Pretrained_v1"
checkpoint = acquire_casnet_checkpoint(weight_cache)
print("CAS-Net checkpoint:", checkpoint)

In [ ]:
from openplaque.independent_coronary_model_anatomy_validation_v1 import write_source_nifti
import SimpleITK as sitk

local_input_dir = Path("/content/openplaque_imagecas_input")
local_input_dir.mkdir(parents=True, exist_ok=True)
source_nii = local_input_dir / "ucla.img.mha"
prediction_nii = Path(OUT) / "external_model" / "ucla.nii.gz"
prediction_nii.parent.mkdir(parents=True, exist_ok=True)
if not source_nii.is_file() or source_nii.stat().st_size == 0:
    print("Local Series 7 MHA missing; regenerating from persistent Drive cache...")
    write_source_nifti(DRIVE_ROOT, source_nii)

if FORCE_EXTERNAL_INFERENCE or not (REUSE_VALID_PREDICTION and prediction_nii.exists()):
    imagecas = Path("/content/ImageCAS-X")
    data_root = Path("/content/imagecas_x_openplaque_data")
    results_root = Path("/content/imagecas_x_openplaque_results")
    run_dir = results_root / "cas_net_openplaque"
    for d in [data_root/"volumes", data_root/"segmentations", data_root/"filelist", results_root, run_dir]:
        d.mkdir(parents=True, exist_ok=True)

    imagecas_input = data_root/"volumes"/"ucla.img.mha"
    if imagecas_input.exists():
        imagecas_input.unlink()
    shutil.copy2(source_nii, imagecas_input)
    if not imagecas_input.is_file() or imagecas_input.stat().st_size == 0:
        raise RuntimeError(f"ImageCAS-X local input copy failed: {imagecas_input}")
    print("ImageCAS-X local input:", imagecas_input, imagecas_input.stat().st_size, "bytes")

    # ImageCAS-X's current VolumeDataset loads a GT mask even during inference.
    # Supply an all-zero mask with identical geometry only to satisfy that loader;
    # it is never used as evidence or for evaluation in this experiment.
    dummy_mask_path = data_root/"segmentations"/"ucla.coronary.mha"
    ref_img = sitk.ReadImage(str(imagecas_input))
    dummy = sitk.Image(ref_img.GetSize(), sitk.sitkUInt8)
    dummy.CopyInformation(ref_img)
    writer = sitk.ImageFileWriter()
    writer.SetFileName(str(dummy_mask_path))
    writer.SetUseCompression(False)
    writer.Execute(dummy)
    if not dummy_mask_path.is_file() or dummy_mask_path.stat().st_size == 0:
        raise RuntimeError(f"Dummy inference mask write failed: {dummy_mask_path}")
    print("ImageCAS-X loader-only zero mask:", dummy_mask_path, dummy_mask_path.stat().st_size, "bytes")

    (data_root/"filelist"/"train.txt").write_text("")
    (data_root/"filelist"/"val.txt").write_text("")
    (data_root/"filelist"/"test.txt").write_text("ucla\n")
    (data_root/"filelist"/"exclude.txt").write_text("")

    cfg_path=imagecas/"configs"/"cas_net_openplaque.json"
    cfg=json.loads((imagecas/"configs"/"cas_net.json").read_text())
    cfg.setdefault("data",{})["volume_suffix"] = ".img.mha"
    cfg.setdefault("data",{})["mask_suffix"] = ".coronary.mha"
    cfg.setdefault("data",{}).setdefault("params",{})["inference_batch_size"] = 1
    cfg.setdefault("training",{})["num_workers"] = 0
    cfg["model"]["checkpoint"] = str(checkpoint)
    cfg_path.write_text(json.dumps(cfg,indent=2))

    os.environ["ImageCAS_X_data_path"] = str(data_root)
    os.environ["ImageCAS_X_results_path"] = str(results_root)

    cmd=[sys.executable,"-u","-m","inference","-c",str(cfg_path),"-r",str(run_dir),"--overwrite"]
    print("Running independent CAS-Net inference locally...")
    print("Command:", " ".join(map(str,cmd)))
    proc = subprocess.Popen(
        cmd, cwd=imagecas, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"ImageCAS-X inference failed with exit code {rc}; full traceback is printed immediately above.")

    produced=run_dir/"predictions"/"ucla.nii.gz"
    if not produced.exists() or produced.stat().st_size == 0:
        raise FileNotFoundError(f"Expected ImageCAS-X prediction not found: {produced}")
    shutil.copy2(produced,prediction_nii)
    if not prediction_nii.exists() or prediction_nii.stat().st_size == 0:
        raise RuntimeError(f"Prediction did not persist to Drive: {prediction_nii}")
else:
    print("Reusing cached independent prediction from Drive:", prediction_nii)

print("Prediction ready:", prediction_nii, prediction_nii.stat().st_size)

In [ ]:
from openplaque.independent_coronary_model_anatomy_validation_v1 import run

summary = run(
    prediction_mask=str(prediction_nii),
    drive_root=DRIVE_ROOT,
    output_dir=OUT,
    model_name="ImageCAS-X CAS-Net",
    model_record="Zenodo 21887809",
)
print(json.dumps(summary["decision"], indent=2))
print("\nStatus:", summary["status"])

In [ ]:
# Compact result review
import pandas as pd
from IPython.display import display, HTML

display(pd.read_csv(Path(OUT)/"path_support_summary.csv"))
display(pd.read_csv(Path(OUT)/"prespecified_gates.csv"))
display(HTML((Path(OUT)/"OPENPLAQUE_INDEPENDENT_CORONARY_MODEL_ANATOMY_VALIDATION_V1_REPORT.html").read_text()))

In [ ]:
# Verify expected deliverables
expected = [
    "run_state.json","summary.json","decision.json","input_provenance.json",
    "path_support_summary.csv","path_point_diagnostics.csv","prespecified_gates.csv",
    "aorta_contact_components.csv","01_independent_model_path_support.png",
    "02_independent_model_topology.png",
    "OPENPLAQUE_INDEPENDENT_CORONARY_MODEL_ANATOMY_VALIDATION_V1_REPORT.html",
    "OPENPLAQUE_INDEPENDENT_CORONARY_MODEL_ANATOMY_VALIDATION_V1_RESULTS.zip",
]
missing=[x for x in expected if not (Path(OUT)/x).exists()]
if missing: raise RuntimeError("Missing outputs: "+str(missing))
print("COMPLETE")
for x in expected: print(Path(OUT)/x)